# Bit-flip rate vs cat size

**Cat size.** `alpha_sq` is `|alpha|^2` (`epsilon_d = 2 alpha_sq g`, `docs/models_ats.md` §2). Saved outputs predate the 2026-09 fix and print twice the current values for the same physical point.

Sweeps `alpha_sq` at fixed `eps_p`, and compares against the perturbative
estimate.

The points are independent (the interaction-frame basis does not move with
`alpha_sq`), so this fans out: **one job, one GPU task per point**, the
autoscaler sizing the pool. The coordinator function runs on a worker and asks
for no resources of its own — each task requests its own GPU.

This is the notebook form of `scripts/sweep_alpha.py`; both call the same
`floquet_lindblad.pipeline.solve_point`. Use the script (via
`scripts/cluster/submit.py sweep_alpha`) for long runs you want to walk away
from — it writes to GCS and survives your laptop closing.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`)
   — these must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access; that is expected and cached.

If you have already run something that imports `ray` in this kernel, restart it
— Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

In [ ]:
import time

N_A, N_B = 30, 11
EPS_P = 0.1
ALPHA_SQ_LIST = list(np.linspace(1.5, 5, 8))

SETTINGS = dict(
    n_a=N_A,
    n_b=N_B,
    eps_p=EPS_P,
    cheb_degree=6,
    m_arnoldi_0=60,
    m_arnoldi=120,
    margin=1e-2,
    want_x_ritz=False,  # only the rates are plotted below
)


def sweep(alpha_sq_list, settings):
    """Coordinator: fans one GPU task out per point, gathers as they land.

    Runs as the driver on a worker (acr.run places it there because it is called
    with no resources of its own); each task asks for its own GPU.
    """
    import ray

    from anb_compute import ray as acr
    from floquet_lindblad.pipeline import solve_point_safe

    refs = {
        acr.submit(solve_point_safe, alpha_sq=float(a), num_gpus=1, num_cpus=8,
                   **settings): float(a)
        for a in alpha_sq_list
    }

    out = []
    pending = list(refs)
    while pending:
        ready, pending = ray.wait(pending, num_returns=1)
        ref, alpha_sq = ready[0], refs[ready[0]]
        try:
            res = ray.get(ref)
        except Exception as exc:  # a task lost to infra, not to numerics
            out.append({"alpha_sq": alpha_sq,
                        "error": f"{type(exc).__name__}: {exc}"})
            continue
        out.append(res)
        print(f"alpha_sq = {alpha_sq:5.2f}  "
              + (res["error"] if "error" in res
                 else f"rate_bf = {res['rate_bf'].real:.6e}"))
    out.sort(key=lambda d: d["alpha_sq"])
    return out


t0 = time.time()
results = acr.run(
    sweep, ALPHA_SQ_LIST, SETTINGS,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
    # No resources here on purpose: the fan-out inside asks for its own.
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
      f"for {len(results)} point(s)")

### Results

In [ ]:
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]

alpha_sq = np.array([r["alpha_sq"] for r in ok])
bf_list = np.array([r["rate_bf"] for r in ok])
res_rel = np.array([r["res"]["res_rel"] for r in ok])

print(f"{'alpha_sq':>9} {'rate_bf':>14} {'res_rel':>11} {'m_a0':>5} {'margin':>9}")
for r in ok:
    print(f"{r['alpha_sq']:9.2f} {r['rate_bf'].real:14.6e} "
          f"{r['res']['res_rel']:11.2e} {r['m_arnoldi_0']:5d} {r['margin']:9.1e}")
for r in bad:
    print(f"{r['alpha_sq']:9.2f}  FAILED: {r['error']}")

### Bit-flip rate vs cat size

In [ ]:
import matplotlib.pyplot as plt

# kappa_1 / kappa_2 come back in each point's `params`, derived on the worker
# from the constants in floquet_lindblad.models.ats -- so they stay the single source of
# truth and are not re-derived here (this client cannot even import ats.py,
# which needs jax + dynamiqs).
kappa_1 = np.array([r["params"]["kappa_1"] for r in ok])
kappa_2 = np.array([r["params"]["kappa_2"] for r in ok])

# Perturbative estimate, kept verbatim from the pre-cluster notebook, including
# the x3000 eyeball normalisation -- it is a shape comparison, not a calibrated
# prefactor. `alpha_sq` is |alpha|^2 (docs/models_ats.md, section 2).
n = alpha_sq
pert_th = (
    kappa_1 * n * np.exp(-4 * n)
    + kappa_1**2 / (2 * kappa_2) * np.exp(-2 * n)
)

fig, ax = plt.subplots(figsize=(5.5, 4.0), constrained_layout=True)
ax.scatter(n, bf_list.real, label="ChebAr", zorder=3)
ax.plot(n, pert_th * 3000, ls=":", c="k", label=r"perturbative $\times$ 3000")
ax.set_yscale("log")
ax.set_xlabel(r"$|\alpha|^2$")
ax.set_ylabel("bit-flip rate (GHz)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()